In [1]:
# %% [markdown]
# ## 7. Inference / Prediction Demo
# Script untuk memprediksi berita baru menggunakan model yang sudah dilatih.

# %%
# %%
import os
import pickle
import re
import numpy as np
from pathlib import Path

# === TAMBAHAN PENTING: PAKSA PAKAI CPU ===
# Taruh ini SEBELUM import tensorflow/keras
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
# =========================================

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ... (lanjutkan sisa kode di bawahnya seperti biasa) ...
# %%
# --- 1. CONFIGURATION ---
MAX_LEN = 200
# Sesuaikan path ini dengan struktur folder kamu
BASE_PATH = Path("../dataset")
PROCESSED_DIR = BASE_PATH / "processed/02_after_FE"
MODEL_DIR = Path("../models")
STOPWORDS_PATH = BASE_PATH / "raw/stopwords_id.txt"

# %%
# --- 2. LOAD RESOURCES ---
print("Loading resources...")

# Load Stopwords
with open(STOPWORDS_PATH, "r", encoding="utf-8") as f:
    stopwords = set([w.strip() for w in f.readlines() if w.strip()])

# Load Tools (Tokenizer & TF-IDF)
tokenizer = pickle.load(open(PROCESSED_DIR / "tokenizer.pkl", "rb"))
tfidf_vectorizer = pickle.load(open(PROCESSED_DIR / "tfidf_vectorizer.pkl", "rb"))

# Load Model (Pilih salah satu, misal BiLSTM)
model_name = "bilstm_model.keras" # Bisa ganti 'lstm_model.keras' atau 'rf_model.pkl'
model = load_model(MODEL_DIR / model_name)

print(f"Resources loaded! Using model: {model_name}")

# %%
# --- 3. UTILITY FUNCTIONS ---

def clean_text(text):
    """Membersihkan teks sama seperti saat training (File 1)"""
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)      # Hapus angka
    text = re.sub(r'[^\w\s]', '', text)  # Hapus tanda baca
    text = re.sub(r'\s+', ' ', text).strip() # Hapus spasi ganda
    
    # Stopwords removal
    tokens = [w for w in text.split() if w not in stopwords]
    return " ".join(tokens)

def prepare_input(text):
    """Mengubah teks mentah menjadi input yang siap masuk model"""
    # 1. Cleaning
    cleaned_text = clean_text(text)
    
    # 2. Sequence Processing (untuk input LSTM/BiLSTM)
    seq = tokenizer.texts_to_sequences([cleaned_text])
    pad = pad_sequences(seq, maxlen=MAX_LEN)
    pad = np.array(pad, dtype='int32') # Pastikan int32
    
    # 3. TF-IDF Processing (untuk input Dense)
    tfidf = tfidf_vectorizer.transform([cleaned_text]).toarray()
    tfidf = np.array(tfidf, dtype='float32') # Pastikan float32
    
    return pad, tfidf

# %%
# --- 4. PREDICTION FUNCTION ---

def predict_hoax(news_text):
    # Siapkan data
    X_pad, X_tfidf = prepare_input(news_text)
    
    # Prediksi
    # Karena model Hybrid butuh 2 input: [Sequence, TFIDF]
    prediction_prob = model.predict([X_pad, X_tfidf], verbose=0)[0][0]
    
    # Ambang batas (Threshold)
    label = "HOAKS" if prediction_prob > 0.5 else "FAKTA"
    confidence = prediction_prob if prediction_prob > 0.5 else 1 - prediction_prob
    
    return label, confidence, prediction_prob

# %%
# --- 5. TEST DENGAN BERITA BARU ---

# Masukkan berita yang mau dites di sini
sample_news = """
Presiden Joko Widodo dikabarkan akan membagikan uang tunai sebesar 
100 juta rupiah kepada seluruh rakyat Indonesia sore ini di Istana Negara.
"""

print("\n" + "="*50)
print("HASIL PREDIKSI")
print("="*50)
print(f"Teks Berita: {sample_news.strip()}")
print("-" * 50)

label, conf, raw_score = predict_hoax(sample_news)

print(f"Prediksi Model : {label}")
print(f"Tingkat Keyakinan: {conf*100:.2f}%")
print(f"Raw Score (0-1)  : {raw_score:.4f}")
print("="*50)

# Coba Berita Lain (Fakta)
sample_news_2 = """
Komisi Pemilihan Umum (KPU) telah menetapkan jadwal debat calon presiden 
dan wakil presiden untuk pemilu 2024 yang akan disiarkan di televisi nasional.
"""

label2, conf2, raw_score2 = predict_hoax(sample_news_2)

print(f"\n[Test 2] Teks: {sample_news_2.strip()[:50]}...")
print(f"Prediksi: {label2} ({conf2*100:.2f}%)")

2025-11-20 07:09:20.450615: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-20 07:09:20.510109: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-20 07:09:21.822966: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Loading resources...


2025-11-20 07:09:23.106237: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-11-20 07:09:23.106473: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
2025-11-20 07:09:23.106482: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2025-11-20 07:09:23.106488: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-11-20 07:09:23.106492: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: MSI
2025-11-20 07:09:23.106495: I external/local_xla/xla/stream_executor/cuda/cuda_diag

Resources loaded! Using model: bilstm_model.keras

HASIL PREDIKSI
Teks Berita: Presiden Joko Widodo dikabarkan akan membagikan uang tunai sebesar 
100 juta rupiah kepada seluruh rakyat Indonesia sore ini di Istana Negara.
--------------------------------------------------
Prediksi Model : HOAKS
Tingkat Keyakinan: 99.75%
Raw Score (0-1)  : 0.9975

[Test 2] Teks: Komisi Pemilihan Umum (KPU) telah menetapkan jadwa...
Prediksi: FAKTA (83.30%)


In [2]:
# %%
import os
import pickle
import re
import numpy as np
from pathlib import Path

# === PAKSA CPU (PENTING) ===
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
# ===========================

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# --- 1. CONFIGURATION ---
MAX_LEN = 200
BASE_PATH = Path("../dataset")
PROCESSED_DIR = BASE_PATH / "processed/02_after_FE"
MODEL_DIR = Path("../models")
STOPWORDS_PATH = BASE_PATH / "raw/stopwords_id.txt"

# --- 2. LOAD RESOURCES ---
print("Loading resources...")

with open(STOPWORDS_PATH, "r", encoding="utf-8") as f:
    stopwords = set([w.strip() for w in f.readlines() if w.strip()])

tokenizer = pickle.load(open(PROCESSED_DIR / "tokenizer.pkl", "rb"))
tfidf_vectorizer = pickle.load(open(PROCESSED_DIR / "tfidf_vectorizer.pkl", "rb"))

# === BAGIAN YANG DIUBAH ===
model_name = "lstm_model.keras"  # <--- Ganti jadi LSTM
# ==========================

model = load_model(MODEL_DIR / model_name)
print(f"Resources loaded! Using model: {model_name}")

# --- 3. & 4. FUNGSI (SAMA SEPERTI SEBELUMNYA) ---
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split() if w not in stopwords]
    return " ".join(tokens)

def prepare_input(text):
    cleaned_text = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned_text])
    pad = pad_sequences(seq, maxlen=MAX_LEN)
    pad = np.array(pad, dtype='int32') 
    tfidf = tfidf_vectorizer.transform([cleaned_text]).toarray()
    tfidf = np.array(tfidf, dtype='float32')
    return pad, tfidf

def predict_hoax(news_text):
    X_pad, X_tfidf = prepare_input(news_text)
    prediction_prob = model.predict([X_pad, X_tfidf], verbose=0)[0][0]
    label = "HOAKS" if prediction_prob > 0.5 else "FAKTA"
    confidence = prediction_prob if prediction_prob > 0.5 else 1 - prediction_prob
    return label, confidence, prediction_prob

# --- 5. TEST ---
sample_news = """
Presiden Joko Widodo dikabarkan akan membagikan uang tunai sebesar 
100 juta rupiah kepada seluruh rakyat Indonesia sore ini di Istana Negara.
"""

print("\n" + "="*50)
print(f"HASIL PREDIKSI MENGGUNAKAN: {model_name}")
print("="*50)

label, conf, raw_score = predict_hoax(sample_news)

print(f"Prediksi Model : {label}")
print(f"Tingkat Keyakinan: {conf*100:.2f}%")
print(f"Raw Score (0-1)  : {raw_score:.4f}")

# Test 2
sample_news_2 = """
Komisi Pemilihan Umum (KPU) telah menetapkan jadwal debat calon presiden 
dan wakil presiden untuk pemilu 2024 yang akan disiarkan di televisi nasional.
"""
label2, conf2, _ = predict_hoax(sample_news_2)
print(f"\n[Test 2] Prediksi: {label2} ({conf2*100:.2f}%)")

Loading resources...
Resources loaded! Using model: lstm_model.keras

HASIL PREDIKSI MENGGUNAKAN: lstm_model.keras
Prediksi Model : HOAKS
Tingkat Keyakinan: 90.74%
Raw Score (0-1)  : 0.9074

[Test 2] Prediksi: FAKTA (80.00%)


In [3]:
# %%
import pickle
import re
import numpy as np
from pathlib import Path

# --- 1. CONFIGURATION ---
BASE_PATH = Path("../dataset")
PROCESSED_DIR = BASE_PATH / "processed/02_after_FE"
MODEL_DIR = Path("../models")
STOPWORDS_PATH = BASE_PATH / "raw/stopwords_id.txt"

# --- 2. LOAD RESOURCES ---
print("Loading resources for Random Forest...")

# Load Stopwords
with open(STOPWORDS_PATH, "r", encoding="utf-8") as f:
    stopwords = set([w.strip() for w in f.readlines() if w.strip()])

# Load TF-IDF Vectorizer (Tokenizer tidak dibutuhkan untuk RF)
tfidf_vectorizer = pickle.load(open(PROCESSED_DIR / "tfidf_vectorizer.pkl", "rb"))

# Load Random Forest Model (.pkl)
model_path = MODEL_DIR / "rf_model.pkl"
with open(model_path, "rb") as f:
    model = pickle.load(f)

print(f"Resources loaded! Using model: Random Forest")

# --- 3. UTILITY FUNCTIONS ---

def clean_text(text):
    """Pembersihan teks standar"""
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [w for w in text.split() if w not in stopwords]
    return " ".join(tokens)

def prepare_input_rf(text):
    """RF HANYA butuh TF-IDF, tidak butuh Sequence/Padding"""
    cleaned_text = clean_text(text)
    
    # Transform ke TF-IDF
    tfidf = tfidf_vectorizer.transform([cleaned_text]).toarray()
    return tfidf

# --- 4. PREDICTION FUNCTION ---

def predict_hoax_rf(news_text):
    # Siapkan data
    X_tfidf = prepare_input_rf(news_text)
    
    # Prediksi Probabilitas
    # predict_proba mengembalikan array: [[prob_0, prob_1]]
    # Kita ambil prob_1 (kemungkinan Hoaks)
    probs = model.predict_proba(X_tfidf)[0]
    prediction_prob = probs[1] 
    
    # Ambang batas
    label = "HOAKS" if prediction_prob > 0.5 else "FAKTA"
    confidence = prediction_prob if prediction_prob > 0.5 else 1 - prediction_prob
    
    return label, confidence, prediction_prob

# --- 5. TEST DENGAN BERITA BARU ---

sample_news = """
Presiden Joko Widodo dikabarkan akan membagikan uang tunai sebesar 
100 juta rupiah kepada seluruh rakyat Indonesia sore ini di Istana Negara.
"""

print("\n" + "="*50)
print("HASIL PREDIKSI: RANDOM FOREST")
print("="*50)

label, conf, raw_score = predict_hoax_rf(sample_news)

print(f"Prediksi Model : {label}")
print(f"Tingkat Keyakinan: {conf*100:.2f}%")
print(f"Raw Score (Probability Hoax): {raw_score:.4f}")

# Test 2 (Berita Fakta)
sample_news_2 = """
Komisi Pemilihan Umum (KPU) telah menetapkan jadwal debat calon presiden 
dan wakil presiden untuk pemilu 2024 yang akan disiarkan di televisi nasional.
"""
label2, conf2, _ = predict_hoax_rf(sample_news_2)
print(f"\n[Test 2] Prediksi: {label2} ({conf2*100:.2f}%)")

Loading resources for Random Forest...
Resources loaded! Using model: Random Forest

HASIL PREDIKSI: RANDOM FOREST
Prediksi Model : HOAKS
Tingkat Keyakinan: 81.50%
Raw Score (Probability Hoax): 0.8150

[Test 2] Prediksi: FAKTA (62.62%)


/home/calista/anaconda3/envs/dl/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/calista/anaconda3/envs/dl/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.7.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
[Parallel(n_jobs=22)]: Using backend ThreadingBackend with 22 concurrent workers.
[Parallel(n_jobs=22)]: Done   6 tasks      | elapsed:    0.0s
[Parallel(n_